# Temporal MLP baselines: global traffic and per-host history

This notebook trains two **flow-feature-only temporal baselines** under the completed ST-GNN factorial protocol. The models use the same stored 30-second graph windows, five seeds, hyperparameters, train/validation/Test 1/Test 2 partitions, early stopping, validation threshold selection, and validation-only champion rule. It produces results compatible with the existing baseline and factorial tables.

- **Global traffic MLP + GRU:** a single recurrent state summarizes the mean, standard deviation, and count of flows in each previous window. Each current flow is scored from its own 32 features and the previous global state. `x`, `edge_index`, and host IDs are ignored.
- **Per-host history MLP + GRU:** each host has a recurrent state built from the mean and count of its own outgoing and incoming flows in previous windows. Each flow is scored from its 32 features and the previous states of its source and destination. Endpoint IDs are required to route flows to hosts; there is **no neighborhood aggregation or message passing**.

Both models score *all* flows in a window before updating memory. The update uses flow features only, never labels. Memory starts empty at every training epoch and at every validation or test split; Test 2 starts empty because its host IDs are scoped to another capture. Thus this is a comparison of recurrent traffic context, not cross-split adaptation. The two temporal heads use the same hidden width and dropout as the original SimpleMLP, but their parameter counts differ and are recorded.

Run top to bottom in Colab. Keep the factorial output folder and graph partitions in Drive. The notebook verifies their hashes before training; each completed seed can be resumed independently. **Test 1 evaluates all five seeds; Test 2 evaluates only the validation-selected seed for each model.** Test 2 uses the existing reused capture and should be reported as exploratory. Do not choose a seed or threshold using either test set.


In [ ]:
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/nids-mitre')
FACTORIAL_ROOT = DRIVE_ROOT / 'results_stgnn_memory_message_factorial_v1'
OUTPUT = DRIVE_ROOT / 'results_temporal_mlp_global_host_factorial_protocol_v1'
CLONE_DIR = Path('/content/temporalgnn-nids-temporal-mlp-utils')
ALLOW_ENVIRONMENT_DIFFERENCES = False
RETRY_FAILED = False
VERIFY_GRAPH_HASHES = True
NUM_WORKERS = 2

reference_path = FACTORIAL_ROOT / 'manifest.json'
assert reference_path.is_file(), f'Missing completed factorial manifest: {reference_path}'
reference_manifest = json.loads(reference_path.read_text())
SOURCE_COMMIT = reference_manifest['source_commit']
assert SOURCE_COMMIT == '2262b53d155f9e8cf5221516f47e4170e6cd0d6e'
reference_software = reference_manifest['software']
for module, requirement in [
    ('torch_geometric', 'torch_geometric==' + reference_software['pyg']),
    ('pyarrow', 'pyarrow'),
]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement], check=True)

if not CLONE_DIR.exists():
    subprocess.run(['git', 'clone', '--no-checkout',
                    'https://github.com/tatipar/temporalgnn-nids.git', str(CLONE_DIR)], check=True)
subprocess.run(['git', '-C', str(CLONE_DIR), 'fetch', '--depth', '1',
                'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(CLONE_DIR), 'checkout', '--detach', SOURCE_COMMIT], check=True)
actual_commit = subprocess.check_output(
    ['git', '-C', str(CLONE_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == SOURCE_COMMIT
sys.path.insert(0, str(CLONE_DIR / 'code/python'))
print('Pinned implementation:', actual_commit)


Mounted at /content/drive
Pinned implementation: 2262b53d155f9e8cf5221516f47e4170e6cd0d6e


## 1. Pin the protocol and define the models

The exact factorial configuration is loaded from its manifest. If Colab's Python/PyTorch/PyG/NumPy/pandas versions differ from the recorded run, the default is to stop. Set `ALLOW_ENVIRONMENT_DIFFERENCES=True` in the setup cell only if you deliberately accept a new software environment; that choice is saved in this run's manifest. The model source is embedded below and copied to the result folder.


In [ ]:
import gc
import hashlib
import os
import platform
import time
import traceback
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch_geometric
from IPython.display import display
from sklearn.metrics import average_precision_score
from torch_geometric.loader import DataLoader

from utils.datasets import NF_IDS_Dataset
import math
from utils.training import train_epoch, evaluate, find_optimal_threshold, set_seed
from utils.experiment import EarlyStopping, NumpyEncoder
from utils.metrics import calculate_metrics_gnn
from utils.evaluation import apply_1sd_rule

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WINDOW_SECONDS = 30
SEEDS = list(reference_manifest['seeds'])
PARAMS = dict(reference_manifest['model_params'])
TRAINING = dict(reference_manifest['training'])
ALERT_BUDGET_FRACTION = float(reference_manifest['ranking_budget_fraction'])
assert SEEDS == [42, 123, 777, 2024, 99]
assert PARAMS == dict(node_dim=16, edge_dim=32, hidden_dim=32,
                      dropout=0.2, output_bias_init=-2.9968)
assert TRAINING == dict(epochs=60, batch_steps=10, learning_rate=0.005,
                        pos_weight=2.0, patience=10, min_delta=0.0001)

# Exact model implementation is embedded, hashed, and saved beside the results.
MODEL_SOURCE = r'''
class FlowMemoryHead(nn.Module):
    """The same two hidden MLP layers as the flow-only SimpleMLP baseline."""

    def __init__(self, input_dim, hidden_dim, dropout, output_bias_init):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1),
        )
        if output_bias_init is not None:
            self.net[-1].bias.data.fill_(output_bias_init)

    def forward(self, features):
        return self.net(features)


class GlobalTrafficMLPGRU(nn.Module):
    """Classify flows with one GRU summary of preceding graph windows only."""

    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2,
                 output_bias_init=None):
        super().__init__()
        self.edge_dim, self.hidden_dim = edge_dim, hidden_dim
        self.memory_cell = nn.GRUCell(2 * edge_dim + 1, hidden_dim)
        self.classifier = FlowMemoryHead(edge_dim + hidden_dim, hidden_dim,
                                         dropout, output_bias_init)
        self.reset_memory()

    def reset_memory(self):
        self.traffic_memory = None

    def detach_all_memory(self):
        if self.traffic_memory is not None:
            self.traffic_memory = self.traffic_memory.detach()

    def forward(self, x, edge_index, edge_attr, global_node_ids):
        n_flows = edge_attr.size(0)
        if n_flows == 0:
            return edge_attr.new_empty((0, 1))
        previous = self.traffic_memory
        if previous is None:
            previous = edge_attr.new_zeros(self.hidden_dim)
        context = previous.unsqueeze(0).expand(n_flows, -1)
        logits = self.classifier(torch.cat((edge_attr, context), dim=1))
        # Update only after all flows in this window have been scored.
        summary = torch.cat((edge_attr.mean(dim=0),
                             edge_attr.std(dim=0, unbiased=False),
                             edge_attr.new_tensor([math.log1p(n_flows)])))
        self.traffic_memory = self.memory_cell(summary.unsqueeze(0),
                                                previous.unsqueeze(0)).squeeze(0)
        return logits


class HostHistoryMLPGRU(nn.Module):
    """Independent host memories from own incident flows; no message passing."""

    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2,
                 output_bias_init=None):
        super().__init__()
        self.edge_dim, self.hidden_dim = edge_dim, hidden_dim
        self.memory_cell = nn.GRUCell(2 * edge_dim + 2, hidden_dim)
        self.classifier = FlowMemoryHead(edge_dim + 2 * hidden_dim,
                                         hidden_dim, dropout, output_bias_init)
        self.reset_memory()

    def reset_memory(self):
        self.host_memory = {}

    def detach_all_memory(self):
        self.host_memory = {gid: state.detach()
                            for gid, state in self.host_memory.items()}

    def forward(self, x, edge_index, edge_attr, global_node_ids):
        n_flows = edge_attr.size(0)
        if n_flows == 0:
            return edge_attr.new_empty((0, 1))
        source_ids = global_node_ids[edge_index[0]].reshape(-1)
        destination_ids = global_node_ids[edge_index[1]].reshape(-1)
        host_ids, inverse = torch.unique(
            torch.cat((source_ids, destination_ids)),
            sorted=True, return_inverse=True)
        host_keys = host_ids.tolist()
        n_hosts = len(host_keys)
        src_index, dst_index = inverse[:n_flows], inverse[n_flows:]
        zero = edge_attr.new_zeros(self.hidden_dim)
        previous = torch.stack([self.host_memory.get(gid, zero)
                                for gid in host_keys], dim=0)
        logits = self.classifier(torch.cat((edge_attr, previous[src_index],
                                             previous[dst_index]), dim=1))

        # Each host sees its own outgoing/incoming flow attributes, not neighbor
        # states. Simultaneous flows use the same pre-window memory for scoring.
        sums_out = edge_attr.new_zeros((n_hosts, self.edge_dim))
        sums_in = edge_attr.new_zeros((n_hosts, self.edge_dim))
        sums_out.index_add_(0, src_index, edge_attr)
        sums_in.index_add_(0, dst_index, edge_attr)
        counts_out = edge_attr.new_zeros((n_hosts, 1))
        counts_in = edge_attr.new_zeros((n_hosts, 1))
        ones = edge_attr.new_ones((n_flows, 1))
        counts_out.index_add_(0, src_index, ones)
        counts_in.index_add_(0, dst_index, ones)
        summary = torch.cat((sums_out / counts_out.clamp_min(1),
                             sums_in / counts_in.clamp_min(1),
                             torch.log1p(counts_out), torch.log1p(counts_in)),
                            dim=1)
        updated = self.memory_cell(summary, previous)
        for row, gid in enumerate(host_keys):
            self.host_memory[gid] = updated[row]
        return logits
'''
MODEL_SHA256 = hashlib.sha256(MODEL_SOURCE.encode()).hexdigest()
exec(compile(MODEL_SOURCE, 'temporal_mlp_models.py', 'exec'), globals())

MODEL_SPECS = {
    'Global traffic MLP-GRU': dict(constructor='GlobalTrafficMLPGRU', folder='global_traffic_mlp_gru', temporal=True),
    'Host history MLP-GRU': dict(constructor='HostHistoryMLPGRU', folder='host_history_mlp_gru', temporal=True),
}
CONSTRUCTORS = dict(GlobalTrafficMLPGRU=GlobalTrafficMLPGRU,
                    HostHistoryMLPGRU=HostHistoryMLPGRU)

def make_model(name):
    return CONSTRUCTORS[MODEL_SPECS[name]['constructor']](**PARAMS)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, cls=NumpyEncoder).encode()).hexdigest()

def write_json(path, value):
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, cls=NumpyEncoder, allow_nan=False))
    os.replace(tmp, path)

def cleanup():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

REFERENCE_MANIFEST_SHA256 = json_digest(reference_manifest)
current_software = dict(python=platform.python_version(), torch=str(torch.__version__),
                         pyg=str(torch_geometric.__version__), numpy=np.__version__, pandas=pd.__version__)
environment = pd.DataFrame({'factorial': reference_software, 'current': current_software})
environment['matches'] = environment['factorial'] == environment['current']
display(environment)
if not ALLOW_ENVIRONMENT_DIFFERENCES:
    assert environment['matches'].all(), (
        'Restore the recorded factorial software environment before training, or explicitly '
        'set ALLOW_ENVIRONMENT_DIFFERENCES=True to record a different environment. '
        'Restart Colab after changing an imported package version.')

OUTPUT.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'none')
print('Training order:', list(MODEL_SPECS))
print('New output:', OUTPUT)


,factorial,current,matches
python,3.13.16,3.13.16,True
torch,2.11.0+cpu,2.11.0+cpu,True
pyg,2.8.1,2.8.1,True
numpy,2.1.3,2.1.3,True
pandas,2.2.3,2.2.3,True


Device: cpu | GPU: none
Training order: ['Global traffic MLP-GRU', 'Host history MLP-GRU']
New output: /content/drive/MyDrive/nids-mitre/results_temporal_mlp_global_host_factorial_protocol_v1


## 2. Verify the four existing graph partitions and freeze the run manifest

The manifest and flow alignment come from the completed factorial run. By default, every `.pt` graph is checked against its saved SHA-256; this can take several minutes on Drive. The loaders are constructed once **before** the first seed is set, matching the corrected factorial protocol. They are sequential (`shuffle=False`, batch size 1); the pinned training utility resets memory at every epoch and detaches it after each ten-window optimizer step.


In [ ]:
SPLITS, datasets, loaders = {}, {}, {}
alignment, graph_inventories, split_manifest = {}, {}, {}
for tag in ('train', 'val', 'test1', 'test2'):
    spec = reference_manifest['splits'][tag]
    root, split, day = Path(spec['root']), spec['directory'], spec['capture_day']
    SPLITS[tag] = (root, split, day)
    assert (root / split).is_dir(), f'Missing graph partition: {root / split}'
    index_path = FACTORIAL_ROOT / 'alignment' / f'{tag}_flows.parquet'
    inventory_path = FACTORIAL_ROOT / 'alignment' / f'{tag}_graphs.csv'
    assert sha256_file(index_path) == spec['alignment_sha256']
    assert sha256_file(inventory_path) == spec['inventory_sha256']
    index = pd.read_parquet(index_path)
    graphs = pd.read_csv(inventory_path)
    dataset = NF_IDS_Dataset(root_dir=str(root), split=split)
    assert dataset.files == graphs['file'].tolist(), f'Graph file list changed: {tag}'
    assert len(dataset) == spec['graphs'] and len(index) == spec['flows']
    assert int(index['label'].sum()) == spec['attacks']
    assert np.array_equal(index['flow_index'].to_numpy(), np.arange(len(index)))
    assert not index.duplicated(['window_id', 'edge_index_in_graph']).any()
    assert index['graph_number'].between(0, len(graphs) - 1).all()
    assert int(graphs['edges'].sum()) == len(index)
    print(f'{tag}: verifying {len(dataset)} saved graphs from {root / split}')
    if VERIFY_GRAPH_HASHES:
        for number, graph_row in enumerate(graphs.itertuples(index=False), start=1):
            graph_path = root / split / graph_row.file
            assert graph_path.stat().st_size == graph_row.bytes, graph_path
            assert sha256_file(graph_path) == graph_row.sha256, f'Graph content changed: {graph_path}'
            if number % 500 == 0:
                print(f'  {tag}: verified {number}/{len(dataset)} files')
    print(f'  {tag}: complete | {len(index):,} flows | {int(index.label.sum()):,} attacks')
    datasets[tag], alignment[tag], graph_inventories[tag] = dataset, index, graphs
    split_manifest[tag] = dict(spec)
    loaders[tag] = DataLoader(dataset, batch_size=1, shuffle=False,
                              num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0,
                              pin_memory=False)

assert pd.Timestamp(split_manifest['train']['last_window']) < pd.Timestamp(split_manifest['val']['first_window'])
assert pd.Timestamp(split_manifest['val']['last_window']) < pd.Timestamp(split_manifest['test1']['first_window'])
for tag, expected in {'test1': (429, 191_679, 3_434),
                       'test2': (2822, 2_135_957, 103_825)}.items():
    spec = split_manifest[tag]
    assert (spec['graphs'], spec['flows'], spec['attacks']) == expected
display(pd.DataFrame(split_manifest).T[
    ['directory', 'capture_day', 'first_window', 'last_window', 'graphs', 'flows', 'attacks']])

parameter_rows = []
for name, spec in MODEL_SPECS.items():
    set_seed(SEEDS[0])
    model = make_model(name)
    parameter_rows.append(dict(model=name, **spec,
                                total_parameters=sum(p.numel() for p in model.parameters()),
                                trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad)))
    del model
parameter_counts = pd.DataFrame(parameter_rows)

source_files = ['models.py', 'training.py', 'datasets.py', 'metrics.py', 'evaluation.py', 'experiment.py']
source_hashes = {f'code/python/utils/{name}': sha256_file(CLONE_DIR / 'code/python/utils' / name)
                 for name in source_files}
for name, digest in source_hashes.items():
    assert digest == reference_manifest['pinned_source_hashes'][name], f'Pinned source differs: {name}'
manifest = dict(
    notebook_version='temporal_mlp_global_host_factorial_protocol_v1', source_commit=SOURCE_COMMIT,
    reference_factorial_root=str(FACTORIAL_ROOT), reference_manifest_sha256=REFERENCE_MANIFEST_SHA256,
    reference_manifest_file_sha256=sha256_file(reference_path),
    seeds=SEEDS, model_params=PARAMS, training=TRAINING, models=MODEL_SPECS,
    model_source_sha256=MODEL_SHA256,
    pinned_source_hashes=source_hashes, splits=split_manifest,
    labels=reference_manifest['labels'], graph_version=reference_manifest['graph_version'],
    ranking_budget_fraction=ALERT_BUDGET_FRACTION,
    threshold_rule=reference_manifest['threshold_rule'],
    champion_rule=reference_manifest['champion_rule'],
    memory_policy='empty_at_each_epoch_and_split; score_before_update; no_warmup',
    software=current_software, sklearn_version=sklearn.__version__,
    environment_matches_factorial=bool(environment['matches'].all()),
    allow_environment_differences=ALLOW_ENVIRONMENT_DIFFERENCES,
    verify_graph_hashes=VERIFY_GRAPH_HASHES,
    loader=dict(batch_size=1, shuffle=False, num_workers=NUM_WORKERS,
                persistent_workers=NUM_WORKERS > 0, pin_memory=False),
)
MANIFEST_SHA256 = json_digest(manifest)
manifest_path = OUTPUT / 'manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, (
        'Output configuration or software changed. Restore the saved settings or use a new OUTPUT.')
else:
    write_json(manifest_path, manifest)
model_copy = OUTPUT / 'temporal_mlp_models.py'
if model_copy.exists():
    assert model_copy.read_text() == MODEL_SOURCE, 'Saved model source changed'
else:
    model_copy.write_text(MODEL_SOURCE)
parameter_counts.to_csv(OUTPUT / 'parameter_counts.csv', index=False)
for name, spec in MODEL_SPECS.items():
    write_json(OUTPUT / f'config_{spec["folder"]}.json', dict(
        model=name, implementation=spec, seeds=SEEDS, model_params=PARAMS, training=TRAINING,
        manifest_sha256=MANIFEST_SHA256,
        parameters=parameter_counts.loc[parameter_counts.model == name].to_dict('records')[0]))
runtime = dict(utc=datetime.now(timezone.utc).isoformat(), device=str(DEVICE),
               gpu=torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
               cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
               manifest_sha256=MANIFEST_SHA256)
write_json(OUTPUT / 'runtime.json', runtime)
reference_runtime_path = FACTORIAL_ROOT / 'runtime.json'
if reference_runtime_path.exists():
    reference_runtime = json.loads(reference_runtime_path.read_text())
    display(pd.DataFrame({'factorial': {k: reference_runtime.get(k) for k in ('device', 'gpu', 'cuda', 'cudnn')},
                          'temporal_mlp': {k: runtime[k] for k in ('device', 'gpu', 'cuda', 'cudnn')}}))
display(parameter_counts)


train: verifying 1998 saved graphs from /content/drive/MyDrive/nids-mitre/dataset_processed/train
  train: verified 500/1998 files
  train: verified 1000/1998 files
  train: verified 1500/1998 files
  train: complete | 1,041,793 flows | 49,564 attacks
val: verifying 428 saved graphs from /content/drive/MyDrive/nids-mitre/dataset_processed/val
  val: complete | 788,535 flows | 31,329 attacks
test1: verifying 429 saved graphs from /content/drive/MyDrive/nids-mitre/dataset_processed/test
  test1: complete | 191,679 flows | 3,434 attacks
test2: verifying 2822 saved graphs from /content/drive/MyDrive/nids-mitre/dataset_processed_thu0103/test2
  test2: verified 500/2822 files
  test2: verified 1000/2822 files
  test2: verified 1500/2822 files
  test2: verified 2000/2822 files
  test2: verified 2500/2822 files
  test2: complete | 2,135,957 flows | 103,825 attacks


,directory,capture_day,first_window,last_window,graphs,flows,attacks
train,train,2018-02-28,2018-02-28T00:12:55.854000,2018-02-28T16:51:25.854000,1998,1041793,49564
val,val,2018-02-28,2018-02-28T16:51:55.854000,2018-02-28T20:25:25.854000,428,788535,31329
test1,test,2018-02-28,2018-02-28T20:25:55.854000,2018-02-28T23:59:55.854000,429,191679,3434
test2,test2,2018-03-01,2018-03-01T00:00:03.289000,2018-03-01T23:30:33.289000,2822,2135957,103825


,factorial,temporal_mlp
device,cpu,cpu
gpu,None,None
cuda,None,None
cudnn,None,None


,model,constructor,folder,temporal,total_parameters,trainable_parameters
0,Global traffic MLP-GRU,GlobalTrafficMLPGRU,global_traffic_mlp_gru,True,12801,12801
1,Host history MLP-GRU,HostHistoryMLPGRU,host_history_mlp_gru,True,13921,13921


## 3. Causality and reset checks

A current flow's score must not depend on other flows from the same window. Changing those other flows should change *later* scores, after the memory update. The checks also confirm that resetting memory restores the same first-window outputs.


In [ ]:
def check_causal_window_scoring(name):
    set_seed(42)
    model = make_model(name).eval()
    x = torch.zeros((3, PARAMS['node_dim']))
    endpoints = torch.tensor([[0, 0], [1, 2]], dtype=torch.long)
    ids = torch.tensor([10, 20, 30], dtype=torch.long)
    first = torch.zeros((2, PARAMS['edge_dim']))
    first[0, 0] = 0.4
    changed = first.clone()
    changed[1, 1] = 8.0
    probe = torch.zeros((1, PARAMS['edge_dim']))
    probe[0, 2] = 0.7
    probe_edges = torch.tensor([[0], [1]], dtype=torch.long)
    with torch.no_grad():
        model.reset_memory()
        first_score = model(x, endpoints, first, ids)[0].clone()
        later_a = model(x, probe_edges, probe, ids).clone()
        model.reset_memory()
        changed_score = model(x, endpoints, changed, ids)[0].clone()
        later_b = model(x, probe_edges, probe, ids).clone()
        model.reset_memory()
        replay = model(x, endpoints, first, ids)[0].clone()
    torch.testing.assert_close(first_score, changed_score, rtol=0, atol=0)
    torch.testing.assert_close(first_score, replay, rtol=0, atol=0)
    assert not torch.equal(later_a, later_b), f'{name}: no effect from preceding window'
    with torch.no_grad():
        model.reset_memory()
        no_node_features = model(torch.ones_like(x), endpoints, first, ids)
    torch.testing.assert_close(no_node_features[0], first_score, rtol=0, atol=0)
    if name == 'Global traffic MLP-GRU':
        with torch.no_grad():
            model.reset_memory()
            no_endpoints = model(x, torch.tensor([[1, 1], [2, 0]]),
                                 first, ids + 100)
        torch.testing.assert_close(no_endpoints[0], first_score, rtol=0, atol=0)
    model.reset_memory()
    model.train()
    model.zero_grad()
    model(x, endpoints, first, ids)
    later = model(x, probe_edges, probe, ids)
    later.sum().backward()
    assert model.memory_cell.weight_ih.grad is not None
    assert model.memory_cell.weight_ih.grad.abs().sum() > 0, 'History gradient did not reach GRU'
    assert len(model.state_dict()) > 0
    model.reset_memory()
    assert (model.traffic_memory is None if hasattr(model, 'traffic_memory')
            else len(model.host_memory) == 0)
    print('PASS:', name, '| same-window causal scoring, temporal effect, reset')

for name in MODEL_SPECS:
    check_causal_window_scoring(name)


PASS: Global traffic MLP-GRU | same-window causal scoring, temporal effect, reset
PASS: Host history MLP-GRU | same-window causal scoring, temporal effect, reset


## 4. Train five seeds per model and calibrate on validation

For every seed, use Adam (`lr=0.005`), weighted BCE (`pos_weight=2`), up to 60 epochs, ten-window truncated backpropagation, and early stopping on validation AUC-PR (patience 10, minimum improvement 0.0001). The best checkpoint is restored, then `max_f1` picks its threshold **only on validation**. Checkpoints, histories, validation scores, and a completion record are saved per seed. A failed seed remains marked failed; set `RETRY_FAILED=True` in the setup cell to rerun it.


In [ ]:
def run_paths(name, seed):
    folder = OUTPUT / MODEL_SPECS[name]['folder'] / f'seed_{seed}'
    return folder, folder / 'checkpoint.pth', folder / 'record.json'

def read_record(name, seed):
    folder, checkpoint, path = run_paths(name, seed)
    if not path.exists():
        return None
    record = json.loads(path.read_text())
    assert record['model'] == name and record['seed'] == seed
    assert record['manifest_sha256'] == MANIFEST_SHA256
    if record['status'] == 'complete':
        assert checkpoint.is_file() and sha256_file(checkpoint) == record['checkpoint_sha256']
        assert (folder / 'val_scores.npz').is_file()
    return record

def train_one(name, seed):
    previous = read_record(name, seed)
    if previous is not None and (previous['status'] == 'complete' or not RETRY_FAILED):
        print(name, 'seed', seed, ': retaining', previous['status'])
        return previous
    folder, checkpoint, record_path = run_paths(name, seed)
    folder.mkdir(parents=True, exist_ok=True)
    set_seed(seed)
    started = time.perf_counter()
    history = []
    temporal = MODEL_SPECS[name]['temporal']
    model = optimizer = criterion = stopper = None
    base_record = dict(model=name, seed=seed, temporal=temporal,
                       run_id=f'{MODEL_SPECS[name]["folder"]}_seed{seed}',
                       manifest_sha256=MANIFEST_SHA256, source_commit=SOURCE_COMMIT)
    try:
        model = make_model(name).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=TRAINING['learning_rate'])
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([TRAINING['pos_weight']], device=DEVICE))
        stopper = EarlyStopping(patience=TRAINING['patience'], min_delta=TRAINING['min_delta'], mode='max')
        for epoch in range(TRAINING['epochs']):
            epoch_start = time.perf_counter()
            train_loss = train_epoch(model, loaders['train'], optimizer, criterion, DEVICE,
                                     is_temporal=temporal, batch_steps=TRAINING['batch_steps'])
            val_loss, labels, probabilities = evaluate(
                model, loaders['val'], criterion, DEVICE, is_temporal=temporal)
            assert np.array_equal(labels, alignment['val'].label.to_numpy())
            val_ap = float(average_precision_score(labels, probabilities))
            if not (np.isfinite([train_loss, val_loss, val_ap]).all() and np.isfinite(probabilities).all()):
                raise FloatingPointError('Nonfinite loss or validation scores')
            improved = stopper(val_ap, model, epoch)
            history.append(dict(epoch=epoch + 1, train_loss=float(train_loss), val_loss=float(val_loss),
                                val_ap=val_ap, seconds=time.perf_counter() - epoch_start))
            pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
            if improved or (epoch + 1) % 10 == 0:
                print(f'{name} seed {seed} | epoch {epoch + 1}: train={train_loss:.4f}, '
                      f'val AP={val_ap:.4f}' + (' *' if improved else ''))
            if stopper.early_stop:
                break
        assert stopper.best_model_state is not None
        model.load_state_dict(stopper.best_model_state, strict=True)
        threshold, labels, probabilities = find_optimal_threshold(
            model, loaders['val'], DEVICE, is_temporal=temporal, strategy='max_f1')
        assert np.array_equal(labels, alignment['val'].label.to_numpy())
        assert np.isfinite(threshold) and np.isfinite(probabilities).all()
        metrics = calculate_metrics_gnn(labels, probabilities, prob_threshold=float(threshold))
        assert np.isfinite(list(metrics.values())).all()
        if temporal:
            model.reset_memory()
        tmp_checkpoint = folder / 'checkpoint.pth.tmp'
        torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, tmp_checkpoint)
        os.replace(tmp_checkpoint, checkpoint)
        checkpoint_hash = sha256_file(checkpoint)
        metadata = dict(split='val', model=name, seed=seed, checkpoint_sha256=checkpoint_hash,
                        threshold=float(threshold), manifest_sha256=MANIFEST_SHA256,
                        alignment_sha256=split_manifest['val']['alignment_sha256'],
                        dataset_sha256=split_manifest['val']['dataset_sha256'])
        tmp_scores = folder / 'val_scores.tmp.npz'
        np.savez_compressed(tmp_scores, flow_index=alignment['val'].flow_index.to_numpy(),
                            probabilities=probabilities.astype(np.float32), metadata=np.array(json.dumps(metadata)))
        os.replace(tmp_scores, folder / 'val_scores.npz')
        record = dict(**base_record, status='complete', threshold=float(threshold),
                      best_epoch=int(stopper.best_epoch), stopped_epoch=len(history),
                      best_val_ap=float(stopper.best_score), val_metrics=metrics,
                      checkpoint_sha256=checkpoint_hash, seconds=time.perf_counter() - started)
        write_json(record_path, record)
        print(name, 'seed', seed, ': complete; best epoch', record['best_epoch'] + 1,
              '| validation AP', metrics['AUC-PR'])
        return record
    except Exception as error:
        record = dict(**base_record, status='failed', error_type=type(error).__name__,
                      error=str(error), traceback=traceback.format_exc(), completed_epochs=len(history),
                      seconds=time.perf_counter() - started)
        write_json(record_path, record)
        print(name, 'seed', seed, ': FAILED:', type(error).__name__, str(error))
        return record
    finally:
        if model is not None and temporal:
            model.reset_memory()
        del model, optimizer, criterion, stopper
        cleanup()

training_status_rows = []
for name in MODEL_SPECS:
    for seed in SEEDS:
        record = train_one(name, seed)
        training_status_rows.append(dict(model=name, seed=seed, status=record['status'],
                                         error=record.get('error', '')))
        pd.DataFrame(training_status_rows).to_csv(OUTPUT / 'training_status.csv', index=False)
training_status = pd.DataFrame(training_status_rows)
display(training_status)


Global traffic MLP-GRU seed 42 | epoch 1: train=0.2070, val AP=0.0690 *
Global traffic MLP-GRU seed 42 | epoch 2: train=0.2120, val AP=0.0693 *
Global traffic MLP-GRU seed 42 | epoch 4: train=0.1990, val AP=0.0732 *
Global traffic MLP-GRU seed 42 | epoch 7: train=0.1964, val AP=0.0771 *
Global traffic MLP-GRU seed 42 | epoch 8: train=0.1975, val AP=0.0785 *
Global traffic MLP-GRU seed 42 | epoch 9: train=0.1972, val AP=0.1762 *
Global traffic MLP-GRU seed 42 | epoch 10: train=0.1962, val AP=0.1889 *
Global traffic MLP-GRU seed 42 | epoch 12: train=0.1950, val AP=0.2083 *
Global traffic MLP-GRU seed 42 | epoch 14: train=0.1942, val AP=0.2106 *
Global traffic MLP-GRU seed 42 | epoch 20: train=0.1944, val AP=0.0715

Optimal Threshold: 0.2088 (Max F1)
Global traffic MLP-GRU seed 42 : complete; best epoch 14 | validation AP 0.2105706963515908
Global traffic MLP-GRU seed 123 | epoch 1: train=0.1987, val AP=0.0305 *
Global traffic MLP-GRU seed 123 | epoch 2: train=0.2061, val AP=0.0698 *
Glob

KeyboardInterrupt: 

The preceding training attempt was interrupted. The following cell resumes from completed records saved in Drive.


In [ ]:
def run_paths(name, seed):
    folder = OUTPUT / MODEL_SPECS[name]['folder'] / f'seed_{seed}'
    return folder, folder / 'checkpoint.pth', folder / 'record.json'

def read_record(name, seed):
    folder, checkpoint, path = run_paths(name, seed)
    if not path.exists():
        return None
    record = json.loads(path.read_text())
    assert record['model'] == name and record['seed'] == seed
    assert record['manifest_sha256'] == MANIFEST_SHA256
    if record['status'] == 'complete':
        assert checkpoint.is_file() and sha256_file(checkpoint) == record['checkpoint_sha256']
        assert (folder / 'val_scores.npz').is_file()
    return record

def train_one(name, seed):
    previous = read_record(name, seed)
    if previous is not None and (previous['status'] == 'complete' or not RETRY_FAILED):
        print(name, 'seed', seed, ': retaining', previous['status'])
        return previous
    folder, checkpoint, record_path = run_paths(name, seed)
    folder.mkdir(parents=True, exist_ok=True)
    set_seed(seed)
    started = time.perf_counter()
    history = []
    temporal = MODEL_SPECS[name]['temporal']
    model = optimizer = criterion = stopper = None
    base_record = dict(model=name, seed=seed, temporal=temporal,
                       run_id=f'{MODEL_SPECS[name]["folder"]}_seed{seed}',
                       manifest_sha256=MANIFEST_SHA256, source_commit=SOURCE_COMMIT)
    try:
        model = make_model(name).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=TRAINING['learning_rate'])
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([TRAINING['pos_weight']], device=DEVICE))
        stopper = EarlyStopping(patience=TRAINING['patience'], min_delta=TRAINING['min_delta'], mode='max')
        for epoch in range(TRAINING['epochs']):
            epoch_start = time.perf_counter()
            train_loss = train_epoch(model, loaders['train'], optimizer, criterion, DEVICE,
                                     is_temporal=temporal, batch_steps=TRAINING['batch_steps'])
            val_loss, labels, probabilities = evaluate(
                model, loaders['val'], criterion, DEVICE, is_temporal=temporal)
            assert np.array_equal(labels, alignment['val'].label.to_numpy())
            val_ap = float(average_precision_score(labels, probabilities))
            if not (np.isfinite([train_loss, val_loss, val_ap]).all() and np.isfinite(probabilities).all()):
                raise FloatingPointError('Nonfinite loss or validation scores')
            improved = stopper(val_ap, model, epoch)
            history.append(dict(epoch=epoch + 1, train_loss=float(train_loss), val_loss=float(val_loss),
                                val_ap=val_ap, seconds=time.perf_counter() - epoch_start))
            pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
            if improved or (epoch + 1) % 10 == 0:
                print(f'{name} seed {seed} | epoch {epoch + 1}: train={train_loss:.4f}, '
                      f'val AP={val_ap:.4f}' + (' *' if improved else ''))
            if stopper.early_stop:
                break
        assert stopper.best_model_state is not None
        model.load_state_dict(stopper.best_model_state, strict=True)
        threshold, labels, probabilities = find_optimal_threshold(
            model, loaders['val'], DEVICE, is_temporal=temporal, strategy='max_f1')
        assert np.array_equal(labels, alignment['val'].label.to_numpy())
        assert np.isfinite(threshold) and np.isfinite(probabilities).all()
        metrics = calculate_metrics_gnn(labels, probabilities, prob_threshold=float(threshold))
        assert np.isfinite(list(metrics.values())).all()
        if temporal:
            model.reset_memory()
        tmp_checkpoint = folder / 'checkpoint.pth.tmp'
        torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, tmp_checkpoint)
        os.replace(tmp_checkpoint, checkpoint)
        checkpoint_hash = sha256_file(checkpoint)
        metadata = dict(split='val', model=name, seed=seed, checkpoint_sha256=checkpoint_hash,
                        threshold=float(threshold), manifest_sha256=MANIFEST_SHA256,
                        alignment_sha256=split_manifest['val']['alignment_sha256'],
                        dataset_sha256=split_manifest['val']['dataset_sha256'])
        tmp_scores = folder / 'val_scores.tmp.npz'
        np.savez_compressed(tmp_scores, flow_index=alignment['val'].flow_index.to_numpy(),
                            probabilities=probabilities.astype(np.float32), metadata=np.array(json.dumps(metadata)))
        os.replace(tmp_scores, folder / 'val_scores.npz')
        record = dict(**base_record, status='complete', threshold=float(threshold),
                      best_epoch=int(stopper.best_epoch), stopped_epoch=len(history),
                      best_val_ap=float(stopper.best_score), val_metrics=metrics,
                      checkpoint_sha256=checkpoint_hash, seconds=time.perf_counter() - started)
        write_json(record_path, record)
        print(name, 'seed', seed, ': complete; best epoch', record['best_epoch'] + 1,
              '| validation AP', metrics['AUC-PR'])
        return record
    except Exception as error:
        record = dict(**base_record, status='failed', error_type=type(error).__name__,
                      error=str(error), traceback=traceback.format_exc(), completed_epochs=len(history),
                      seconds=time.perf_counter() - started)
        write_json(record_path, record)
        print(name, 'seed', seed, ': FAILED:', type(error).__name__, str(error))
        return record
    finally:
        if model is not None and temporal:
            model.reset_memory()
        del model, optimizer, criterion, stopper
        cleanup()

training_status_rows = []
for name in MODEL_SPECS:
    for seed in SEEDS:
        record = train_one(name, seed)
        training_status_rows.append(dict(model=name, seed=seed, status=record['status'],
                                         error=record.get('error', '')))
        pd.DataFrame(training_status_rows).to_csv(OUTPUT / 'training_status.csv', index=False)
training_status = pd.DataFrame(training_status_rows)
display(training_status)


Global traffic MLP-GRU seed 42 : retaining complete
Global traffic MLP-GRU seed 123 : retaining complete
Global traffic MLP-GRU seed 777 : retaining complete
Global traffic MLP-GRU seed 2024 : retaining complete
Global traffic MLP-GRU seed 99 : retaining complete
Host history MLP-GRU seed 42 : retaining complete
Host history MLP-GRU seed 123 : retaining complete
Host history MLP-GRU seed 777 : retaining complete
Host history MLP-GRU seed 2024 | epoch 1: train=0.2001, val AP=0.2538 *
Host history MLP-GRU seed 2024 | epoch 2: train=0.1941, val AP=0.3220 *
Host history MLP-GRU seed 2024 | epoch 3: train=0.1947, val AP=0.4813 *
Host history MLP-GRU seed 2024 | epoch 6: train=0.1080, val AP=0.5228 *
Host history MLP-GRU seed 2024 | epoch 7: train=0.1037, val AP=0.5718 *
Host history MLP-GRU seed 2024 | epoch 8: train=0.0887, val AP=0.6115 *
Host history MLP-GRU seed 2024 | epoch 9: train=0.0811, val AP=0.6375 *
Host history MLP-GRU seed 2024 | epoch 10: train=0.0782, val AP=0.6414 *
Host hi

,model,seed,status,error
0,Global traffic MLP-GRU,42,complete,
1,Global traffic MLP-GRU,123,complete,
2,Global traffic MLP-GRU,777,complete,
3,Global traffic MLP-GRU,2024,complete,
4,Global traffic MLP-GRU,99,complete,
5,Host history MLP-GRU,42,complete,
6,Host history MLP-GRU,123,complete,
7,Host history MLP-GRU,777,complete,
8,Host history MLP-GRU,2024,complete,
9,Host history MLP-GRU,99,complete,


## 5. Freeze validation-selected champions

Require all five completed seeds for each model. Apply the same one-standard-deviation rule on validation F1; among eligible seeds, choose the earliest best epoch and break ties by validation F1. The chosen seed and threshold are frozen before either test evaluation.


In [ ]:
validation_rows = []
for name in MODEL_SPECS:
    for seed in SEEDS:
        record = read_record(name, seed)
        row = dict(Architecture=name, Raw_Dir_Name=MODEL_SPECS[name]['folder'], Seed=seed,
                   status=record['status'] if record else 'missing')
        if row['status'] == 'complete':
            row.update(F1_Score=record['val_metrics']['F1'], Best_Epoch=record['best_epoch'],
                       Best_Epoch_One_Based=record['best_epoch'] + 1,
                       Optimal_Threshold=record['threshold'], Val_AP=record['val_metrics']['AUC-PR'],
                       checkpoint_sha256=record['checkpoint_sha256'])
        else:
            row['error'] = record.get('error', '') if record else 'No record'
        validation_rows.append(row)
validation = pd.DataFrame(validation_rows)
validation.to_csv(OUTPUT / 'validation_all_seeds.csv', index=False)
champion_rows = []
for name in MODEL_SPECS:
    group = validation.loc[validation.Architecture == name]
    if len(group) == len(SEEDS) and group.status.eq('complete').all():
        selected = apply_1sd_rule(group).iloc[0].to_dict()
        selected.update(selection_status='selected',
                        F1_cutoff=float(group.F1_Score.max() - group.F1_Score.std()),
                        manifest_sha256=MANIFEST_SHA256)
    else:
        selected = dict(Architecture=name, selection_status='incomplete_no_champion',
                        manifest_sha256=MANIFEST_SHA256)
    champion_rows.append(selected)
champions = pd.DataFrame(champion_rows)
champions.to_csv(OUTPUT / 'champions_by_validation.csv', index=False)
champion_seeds = {row['Architecture']: int(row['Seed']) for row in champion_rows
                  if row['selection_status'] == 'selected'}
write_json(OUTPUT / 'frozen_selection.json', dict(
    manifest_sha256=MANIFEST_SHA256, validation_sha256=sha256_file(OUTPUT / 'validation_all_seeds.csv'),
    champions_sha256=sha256_file(OUTPUT / 'champions_by_validation.csv'),
    champion_seeds=champion_seeds, test2_status='exploratory_reused_capture'))
display(validation)
display(champions)


                       SELECTION OF CHAMPIONS (RULE 1-SD)                       

Architecture: GLOBAL TRAFFIC MLP-GRU (Seeds analyzed: 5)
  -> F1 Range   : Max = 0.2221 | Std = 0.0094
  -> Zone 1-SD  : Models with F1 >= 0.2127
  -> Candidates : 2 seed(s) entered the safe zone.
     CHAMPION   : Seed 42 | F1: 0.2221 | Training epochs: 13
                       SELECTION OF CHAMPIONS (RULE 1-SD)                       

Architecture: HOST HISTORY MLP-GRU (Seeds analyzed: 5)
  -> F1 Range   : Max = 0.7795 | Std = 0.0252
  -> Zone 1-SD  : Models with F1 >= 0.7544
  -> Candidates : 1 seed(s) entered the safe zone.
     CHAMPION   : Seed 2024 | F1: 0.7795 | Training epochs: 19


,Architecture,Raw_Dir_Name,Seed,status,F1_Score,Best_Epoch,Best_Epoch_One_Based,Optimal_Threshold,Val_AP,checkpoint_sha256
0,Global traffic MLP-GRU,global_traffic_mlp_gru,42,complete,0.222117,13,14,0.208754,0.210571,56319491fc6b092fa8dcd235b4cc7d46607069afd58408...
1,Global traffic MLP-GRU,global_traffic_mlp_gru,123,complete,0.218696,13,14,0.198220,0.201961,6d9c784bfe8b541848abde524eb9f751641cbd61f7ae0b...
2,Global traffic MLP-GRU,global_traffic_mlp_gru,777,complete,0.201933,18,19,0.209693,0.202121,9f3b8b889ce6011082f7aed8b9019a50b03a9dd1079338...
3,Global traffic MLP-GRU,global_traffic_mlp_gru,2024,complete,0.203069,11,12,0.211455,0.205762,dfe841afe8cd76dad9b0b28b21327f2a4b8be29d24b230...
4,Global traffic MLP-GRU,global_traffic_mlp_gru,99,complete,0.205775,16,17,0.205353,0.204716,11132d03d9f93b39a4fa8537e3a7e4d5ed3f9897f1ae61...
5,Host history MLP-GRU,host_history_mlp_gru,42,complete,0.731811,17,18,0.246683,0.750953,a9ae463666e26a3b8cdf0780becaf8bbc347487a1c448c...
6,Host history MLP-GRU,host_history_mlp_gru,123,complete,0.720996,21,22,0.729708,0.716923,9ca1560400fdda76f27627c68d753a00d3b44ad3ded687...
7,Host history MLP-GRU,host_history_mlp_gru,777,complete,0.746756,28,29,0.046690,0.744046,a3c7fcbd35c83ff588cbc085c9b9803b8a6d2e2be1cc60...
8,Host history MLP-GRU,host_history_mlp_gru,2024,complete,0.779541,19,20,0.842090,0.798113,a9191595c33ebbfcf56d80f690f161bbd7a0310ec37824...
9,Host history MLP-GRU,host_history_mlp_gru,99,complete,0.717723,20,21,0.785627,0.726111,5ce13bea0e0580b211b28e4a999008e2f35b68902e3b09...


,Architecture,Raw_Dir_Name,Seed,status,F1_Score,Best_Epoch,Best_Epoch_One_Based,Optimal_Threshold,Val_AP,checkpoint_sha256,selection_status,F1_cutoff,manifest_sha256
0,Global traffic MLP-GRU,global_traffic_mlp_gru,42,complete,0.222117,13,14,0.208754,0.210571,56319491fc6b092fa8dcd235b4cc7d46607069afd58408...,selected,0.212724,3fe308d6f7e7640f42cfa7ffbbdc5a2e207c8a6a4f1a5c...
1,Host history MLP-GRU,host_history_mlp_gru,2024,complete,0.779541,19,20,0.842090,0.798113,a9191595c33ebbfcf56d80f690f161bbd7a0310ec37824...,selected,0.754383,3fe308d6f7e7640f42cfa7ffbbdc5a2e207c8a6a4f1a5c...


## 6. Infer aligned flow scores and evaluate Test 1 / Test 2

The flow index is checked against labels and endpoint IDs window by window. Inference resets memory at the start of each split. The saved threshold from validation is applied with the pinned `score > threshold` rule. Besides standard confusion-matrix metrics, the tables contain detected attacked sources, false alerts, and workload summaries. Test 1 contains every seed; Test 2 contains only each model's validation champion.


In [ ]:
@torch.no_grad()
def aligned_scores(model, tag, checkpoint, threshold, folder, model_name, seed):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    cache = folder / f'{tag}_scores.npz'
    checkpoint_hash = sha256_file(checkpoint)
    index = alignment[tag]
    temporal = MODEL_SPECS[model_name]['temporal']
    metadata = dict(split=tag, model=model_name, seed=int(seed),
                    checkpoint_sha256=checkpoint_hash, threshold=float(threshold),
                    manifest_sha256=MANIFEST_SHA256,
                    alignment_sha256=split_manifest[tag]['alignment_sha256'],
                    dataset_sha256=split_manifest[tag]['dataset_sha256'])
    if cache.exists():
        with np.load(cache, allow_pickle=False) as saved:
            assert json.loads(str(saved['metadata'].item())) == metadata, f'Stale cache: {cache}'
            assert np.array_equal(saved['flow_index'], index.flow_index.to_numpy())
            probabilities = saved['probabilities'].copy()
    else:
        model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True), strict=True)
        model.to(DEVICE).eval()
        if temporal:
            model.reset_memory()
        probabilities = np.empty(len(index), dtype=np.float32)
        offset = 0
        for graph_number, graph in enumerate(loaders[tag]):
            n = int(graph.y.numel())
            if graph.x.shape[0] == 0:
                assert n == 0
                continue
            expected = index.iloc[offset:offset + n]
            assert np.array_equal(graph.y.reshape(-1).numpy().astype(np.int8), expected.label.to_numpy())
            gids = graph.global_node_ids.reshape(-1)
            assert np.array_equal(gids[graph.edge_index[0]].numpy(), expected.source_gid.to_numpy())
            assert np.array_equal(gids[graph.edge_index[1]].numpy(), expected.destination_gid.to_numpy())
            assert expected.graph_number.eq(graph_number).all()
            graph = graph.to(DEVICE)
            if temporal:
                logits = model(graph.x, graph.edge_index, graph.edge_attr, graph.global_node_ids).reshape(-1)
            else:
                logits = model(graph.x, graph.edge_index, graph.edge_attr).reshape(-1)
            probabilities[offset:offset + n] = torch.sigmoid(logits).cpu().numpy()
            offset += n
            if (graph_number + 1) % 500 == 0:
                print(model_name, tag, ': inferred', graph_number + 1, 'windows')
        assert offset == len(index)
        if temporal:
            model.reset_memory()
        assert np.isfinite(probabilities).all()
        tmp = folder / f'{tag}_scores.tmp.npz'
        np.savez_compressed(tmp, flow_index=index.flow_index.to_numpy(),
                            probabilities=probabilities, metadata=np.array(json.dumps(metadata)))
        os.replace(tmp, cache)
    assert len(probabilities) == len(index)
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    return probabilities, metadata


In [ ]:
def summarize_scores(tag, model_name, seed, probabilities, threshold, folder, metadata):
    index = alignment[tag]
    y = index.label.to_numpy()
    source = index.source_gid.to_numpy()
    alert = probabilities > threshold
    metrics = calculate_metrics_gnn(y, probabilities, prob_threshold=threshold)
    times = pd.to_datetime(index.window_start_ns, unit='ns')
    # Include empty boundary windows in the observation interval and censoring time.
    split_start_ns = pd.Timestamp(graph_inventories[tag].window_start.iloc[0]).value
    end_ns = pd.Timestamp(graph_inventories[tag].window_start.iloc[-1]).value + WINDOW_SECONDS * 1_000_000_000
    duration_days = (end_ns - split_start_ns) / (86400 * 1e9)
    flow = pd.DataFrame(dict(source_gid=source, window_id=index.window_id.to_numpy(),
                              window_start_ns=index.window_start_ns.to_numpy(),
                              attack=y.astype(bool), alert=alert, tp=alert & (y == 1),
                              fp=alert & (y == 0)))
    source_windows = flow.groupby(['source_gid', 'window_id'], sort=False).agg(
        attack=('attack', 'any'), alert=('alert', 'any'), tp=('tp', 'any'))
    sources = []
    for gid, group in flow[flow.attack].groupby('source_gid', sort=True):
        first_ns = int(group.window_start_ns.min())
        hits = group.loc[group.tp, 'window_start_ns']
        detected = len(hits) > 0
        first_hit_ns = int(hits.min()) if detected else None
        followup = (end_ns - first_ns) / 60e9
        delay = (first_hit_ns - first_ns) / 60e9 if detected else None
        sources.append(dict(
            split=tag, model=model_name, seed=seed, capture_day=SPLITS[tag][2], source_gid=int(gid),
            attack_flows=len(group), attack_windows=group.window_id.nunique(),
            first_attack_window_ns=first_ns, first_hit_window_ns=first_hit_ns,
            event_observed=detected, nominal_first_hit_minutes=delay,
            nominal_followup_minutes=followup,
            nominal_event_or_censor_minutes=delay if detected else followup,
            attack_windows_detected=group.loc[group.tp, 'window_id'].nunique()))
    # Nullable integer timestamps preserve nanosecond keys for censored sources.
    first_hits = pd.array([row['first_hit_window_ns'] for row in sources], dtype='Int64')
    sources = pd.DataFrame(sources)
    sources['first_hit_window_ns'] = first_hits
    sources.to_csv(folder / f'{tag}_source_detection.csv', index=False)
    source_windows.reset_index().to_csv(folder / f'{tag}_source_windows.csv', index=False)
    daily = pd.DataFrame(dict(calendar_day=times.dt.strftime('%Y-%m-%d'),
                               false_alerts=flow.fp, alerts=alert, true_positives=flow.tp))
    daily = daily.groupby('calendar_day').sum().reset_index()
    daily.to_csv(folder / f'{tag}_daily_workload.csv', index=False)

    k = max(1, int(np.floor(ALERT_BUDGET_FRACTION * len(y))))
    selected = np.argsort(-probabilities, kind='stable')[:k]
    budget_tp = int(y[selected].sum())
    detected_at_budget = np.unique(source[selected][y[selected] == 1]).size
    row = dict(
        split=tag, model=model_name, seed=seed, status='complete',
        threshold=float(threshold), flows=len(y), attacks=int(y.sum()), prevalence=float(y.mean()),
        attacked_sources=len(sources), sources_detected=int(sources.event_observed.sum()),
        sources_never_detected=int((~sources.event_observed).sum()),
        source_detection_fraction=float(sources.event_observed.mean()),
        detected_first_attack_window=int(sources.nominal_first_hit_minutes.eq(0).sum()),
        detected_within_5_nominal_min=int(sources.nominal_first_hit_minutes.le(5).sum()),
        detected_within_30_nominal_min=int(sources.nominal_first_hit_minutes.le(30).sum()),
        alerts=int(alert.sum()), false_alerts=int(metrics['FP']),
        observed_nominal_days=duration_days,
        false_alerts_per_nominal_24h=metrics['FP'] / duration_days,
        alerted_source_windows=int(source_windows.alert.sum()),
        source_windows_without_attack_alerted=int((source_windows.alert & ~source_windows.attack).sum()),
        budget_fraction=ALERT_BUDGET_FRACTION, budget_alerts=k,
        TP_at_budget=budget_tp, FP_at_budget=k - budget_tp,
        recall_at_budget=budget_tp / int(y.sum()),
        source_detection_fraction_at_budget=detected_at_budget / len(sources),
        checkpoint_sha256=metadata['checkpoint_sha256'],
        alignment_sha256=metadata['alignment_sha256'], **metrics)
    write_json(folder / f'{tag}_metrics.json', row)
    return row


In [ ]:
def evaluate_baseline(tag, name, seed):
    record = read_record(name, seed)
    if record is None or record['status'] != 'complete':
        return dict(split=tag, model=name, seed=seed,
                    status=record['status'] if record else 'missing',
                    error=record.get('error', '') if record else 'No record')
    folder, checkpoint, _ = run_paths(name, seed)
    model = make_model(name)
    try:
        probabilities, metadata = aligned_scores(
            model, tag, checkpoint, record['threshold'], folder, name, seed)
        return summarize_scores(tag, name, seed, probabilities, record['threshold'], folder, metadata)
    finally:
        del model
        cleanup()

test1_rows = []
for name in MODEL_SPECS:
    for seed in SEEDS:
        test1_rows.append(evaluate_baseline('test1', name, seed))
        pd.DataFrame(test1_rows).to_csv(OUTPUT / 'test1_all_seeds.csv', index=False)
baseline_test1 = pd.DataFrame(test1_rows)
test2_rows = []
for name in MODEL_SPECS:
    if name in champion_seeds:
        test2_rows.append(evaluate_baseline('test2', name, champion_seeds[name]))
    else:
        test2_rows.append(dict(split='test2', model=name, seed=None, status='incomplete_no_champion'))
    pd.DataFrame(test2_rows).to_csv(OUTPUT / 'test2_champions.csv', index=False)
baseline_test2 = pd.DataFrame(test2_rows)
for frame in (baseline_test1, baseline_test2):
    display(frame.reindex(columns=['split', 'model', 'seed', 'status', 'AUC-PR', 'F1', 'FPR',
                                    'sources_detected', 'TP_at_budget', 'FP',
                                    'source_windows_without_attack_alerted']).round(5))


Global traffic MLP-GRU test2 : inferred 1500 windows
Global traffic MLP-GRU test2 : inferred 2000 windows
Global traffic MLP-GRU test2 : inferred 2500 windows
Host history MLP-GRU test2 : inferred 1500 windows
Host history MLP-GRU test2 : inferred 2000 windows
Host history MLP-GRU test2 : inferred 2500 windows


,split,model,seed,status,AUC-PR,F1,FPR,sources_detected,TP_at_budget,FP,source_windows_without_attack_alerted
0,test1,Global traffic MLP-GRU,42,complete,0.04628,0.00103,0.00243,1,51,458,429
1,test1,Global traffic MLP-GRU,123,complete,0.04561,0.00000,0.00092,0,50,173,157
2,test1,Global traffic MLP-GRU,777,complete,0.04926,0.00000,0.00100,0,71,189,182
3,test1,Global traffic MLP-GRU,2024,complete,0.04591,0.00000,0.00123,0,65,232,218
4,test1,Global traffic MLP-GRU,99,complete,0.04851,0.00000,0.00078,0,67,146,131
5,test1,Host history MLP-GRU,42,complete,0.69957,0.72692,0.00623,10,1646,1172,380
6,test1,Host history MLP-GRU,123,complete,0.67032,0.70691,0.00736,11,1613,1386,273
7,test1,Host history MLP-GRU,777,complete,0.70190,0.63359,0.01240,13,1646,2335,442
8,test1,Host history MLP-GRU,2024,complete,0.70850,0.72352,0.00609,11,1682,1146,87
9,test1,Host history MLP-GRU,99,complete,0.71362,0.73309,0.00543,9,1650,1023,159


,split,model,seed,status,AUC-PR,F1,FPR,sources_detected,TP_at_budget,FP,source_windows_without_attack_alerted
0,test2,Global traffic MLP-GRU,42,complete,0.39884,0.45764,0.00181,72,20810,3670,3060
1,test2,Host history MLP-GRU,2024,complete,0.66240,0.66694,0.01646,34,18763,33453,741


## 7. Inspect results across seeds and compare with the existing MLP

The Test 1 table below reports mean and sample standard deviation across five seeds; it does not select a test winner. Test 2 shows the frozen validation champions. The original flow-only MLP and factorial M1_P1 are loaded only for a descriptive comparison when their tables are available. Their checkpoints and metrics are never modified.


In [ ]:
assert baseline_test1.groupby('model')['seed'].nunique().eq(len(SEEDS)).all(), 'Incomplete Test 1 runs'
assert baseline_test1.status.eq('complete').all(), 'Incomplete Test 1 runs'
assert baseline_test2.status.eq('complete').all(), 'Incomplete Test 2 runs'
metrics_to_show = ['AUC-PR', 'AUC-ROC', 'Precision', 'Recall', 'F1', 'F2', 'FPR',
                   'sources_detected', 'false_alerts']
summary = baseline_test1.groupby('model')[metrics_to_show].agg(['mean', 'std'])
summary.to_csv(OUTPUT / 'test1_mean_std.csv')
display(summary.round(5))
display(baseline_test2[['model', 'seed', 'threshold', 'AUC-PR', 'F1', 'FPR',
                        'sources_detected', 'false_alerts']].round(5))

reference_tables = []
for root, label, model_name in [
    (DRIVE_ROOT / 'results_baseline_retraining_factorial_protocol_v1',
     'Flow-only MLP', 'MLP'),
    (FACTORIAL_ROOT, 'M1_P1 (corrected ST-GNN)', 'M1_P1'),
]:
    for tag, filename in [('test1', 'test1_all_seeds.csv'),
                          ('test2', 'test2_champions.csv')]:
        path = root / filename
        if path.exists():
            table = pd.read_csv(path)
            subset = table.loc[table.model.eq(model_name) & table.status.eq('complete')].copy()
            if len(subset):
                subset['comparison_model'] = label
                subset['split'] = tag
                reference_tables.append(subset)
comparison = pd.concat([baseline_test1.assign(comparison_model=baseline_test1.model),
                        baseline_test2.assign(comparison_model=baseline_test2.model),
                        *reference_tables], ignore_index=True)
comparison.to_csv(OUTPUT / 'descriptive_comparison.csv', index=False)
display(comparison.groupby(['split', 'comparison_model'])[['AUC-PR', 'F1', 'FPR']]
        .agg(['mean', 'std']).round(5))
print('Results:', OUTPUT)


AUC-PR           AUC-ROC          Precision           \
                           mean      std     mean      std      mean      std   
model                                                                           
Global traffic MLP-GRU  0.04711  0.00166  0.81329  0.00311   0.00087  0.00194   
Host history MLP-GRU    0.69878  0.01685  0.96728  0.00779   0.65814  0.07277   

                         Recall                F1                F2           \
                           mean      std     mean      std     mean      std   
model                                                                          
Global traffic MLP-GRU  0.00012  0.00026  0.00021  0.00046  0.00014  0.00032   
Host history MLP-GRU    0.76383  0.01086  0.70480  0.04098  0.73833  0.01423   

                            FPR          sources_detected           \
                           mean      std             mean      std   
model                                                                
Global traffic MLP-GRU  0.00127  0.00067              0.2  0.44721   
Host history MLP-GRU    0.00750  0.00283             10.8  1.48324   

                       false_alerts             
                               mean        std  
model                                           
Global traffic MLP-GRU        239.6  126.00913  
Host history MLP-GRU         1412.4  532.04351

,model,seed,threshold,AUC-PR,F1,FPR,sources_detected,false_alerts
0,Global traffic MLP-GRU,42,0.20875,0.39884,0.45764,0.00181,72,3670
1,Host history MLP-GRU,2024,0.84209,0.66240,0.66694,0.01646,34,33453


AUC-PR                F1               FPR  \
                                   mean      std     mean      std     mean   
split comparison_model                                                        
test1 Flow-only MLP             0.05098  0.00776  0.00124  0.00171  0.00235   
      Global traffic MLP-GRU    0.04711  0.00166  0.00021  0.00046  0.00127   
      Host history MLP-GRU      0.69878  0.01685  0.70480  0.04098  0.00750   
      M1_P1 (corrected ST-GNN)  0.68805  0.02084  0.72843  0.02881  0.00672   
test2 Flow-only MLP             0.39610      NaN  0.47217      NaN  0.00224   
      Global traffic MLP-GRU    0.39884      NaN  0.45764      NaN  0.00181   
      Host history MLP-GRU      0.66240      NaN  0.66694      NaN  0.01646   
      M1_P1 (corrected ST-GNN)  0.79885      NaN  0.71569      NaN  0.01776   

                                         
                                    std  
split comparison_model                   
test1 Flow-only MLP             0.00152  
      Global traffic MLP-GRU    0.00067  
      Host history MLP-GRU      0.00283  
      M1_P1 (corrected ST-GNN)  0.00170  
test2 Flow-only MLP                 NaN  
      Global traffic MLP-GRU        NaN  
      Host history MLP-GRU          NaN  
      M1_P1 (corrected ST-GNN)      NaN

Results: /content/drive/MyDrive/nids-mitre/results_temporal_mlp_global_host_factorial_protocol_v1
